In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

### desercion_por_especialidad

In [0]:
df_citas = spark.table(f"{catalogo}.{esquema_source}.citas_detalle")

In [0]:
df_desercion_esp = df_citas.groupBy("nombre_especialidad", "area_especialidad")\
    .agg(
        count("id_cita").alias("total_citas"),
        count(when(col("estado") == "Atendido", 1)).alias("total_atendidos"),
        count(when(col("estado") == "Inasistencia", 1)).alias("total_inasistencias"),
        count(when(col("estado") == "Cancelado", 1)).alias("total_cancelados")
    )\
    .withColumn("tasa_desercion", 
        round((col("total_inasistencias") / col("total_citas")) * 100, 2))

In [0]:
df_desercion_esp.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.desercion_por_especialidad")

### desercion_por_seguro

In [0]:
df_desercion_seguro = df_citas.groupBy("tipo_seguro")\
    .agg(
        count("id_cita").alias("total_citas"),
        count(when(col("estado") == "Inasistencia", 1)).alias("total_inasistencias")
    )\
    .withColumn("tasa_desercion",
        round((col("total_inasistencias") / col("total_citas")) * 100, 2))

In [0]:
df_desercion_seguro.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.desercion_por_seguro")

### ingresos_por_especialidad

In [0]:
df_atenciones = spark.table(f"{catalogo}.{esquema_source}.atenciones_detalle")

In [0]:
df_ingresos = df_atenciones.groupBy("nombre_especialidad", "area_especialidad")\
    .agg(
        count("id_atencion").alias("total_atenciones"),
        sum("monto_pagado").cast("double").alias("ingreso_total"),
        round(avg("monto_pagado"), 2).alias("ingreso_promedio")
    )

In [0]:
df_ingresos.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.ingresos_por_especialidad")

### pacientes_recurrentes

In [0]:
df_pacientes_rec = df_citas.groupBy("id_paciente", "nombre_paciente", "distrito_paciente", "tipo_seguro")\
    .agg(
        count("id_cita").alias("total_citas"),
        count(when(col("estado") == "Atendido", 1)).alias("total_atenciones"),
        count(when(col("estado") == "Inasistencia", 1)).alias("total_inasistencias")
    )\
    .withColumn("tasa_desercion",
        round((col("total_inasistencias") / col("total_citas")) * 100, 2))\
    .filter(col("total_citas") >= 2)\
    .orderBy(col("tasa_desercion").desc())

In [0]:
df_pacientes_rec.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.pacientes_recurrentes")